In [0]:
from pyspark.sql.functions import col, to_timestamp, when, to_utc_timestamp

print("1. Reading Bronze Historical Weather...")
bronze_weather = spark.table("aviation_project.bronze_historical_weather")

In [0]:
# ---------------------------------------------------------
# Part 1: Feature Mapping and WMO Translation
# ---------------------------------------------------------
print("2. Mapping Timezones and Translating to UTC...")
# Define the dynamic timezone map
timezone_expr = when(col("airport_code").isin("ATL", "CLT", "LGA", "DCA", "MIA"), "America/New_York") \
                .when(col("airport_code").isin("LIT", "DFW", "ORD", "IAH", "DAL", "STL"), "America/Chicago") \
                .when(col("airport_code") == "DEN", "America/Denver") \
                .when(col("airport_code") == "LAS", "America/Los_Angeles") \
                .otherwise("UTC")

silver_weather = bronze_weather.select(
    # Parse the local string, then immediately shift it to UTC based on the airport
    to_utc_timestamp(to_timestamp(col("time"), "yyyy-MM-dd'T'HH:mm"), timezone_expr).alias("weather_timestamp_utc"),
    
    col("airport_code"),
    col("temperature_2m").cast("double").alias("temperature_f"),
    col("wind_speed_10m").cast("double").alias("wind_speed_mph"),
    col("precipitation").cast("double").alias("precipitation_inch"),
    
    when(col("weather_code") == 0, "Clear")
    .when(col("weather_code").isin(1, 2, 3), "Cloudy")
    .when(col("weather_code").isin(45, 48), "Fog")
    .when(col("weather_code").isin(51, 53, 55, 56, 57), "Drizzle")
    .when(col("weather_code").isin(61, 63, 65, 66, 67, 80, 81, 82), "Rain")
    .when(col("weather_code").isin(71, 73, 75, 77, 85, 86), "Snow")
    .when(col("weather_code").isin(95, 96, 99), "Thunderstorm")
    .otherwise("Unknown").alias("condition")
)

silver_weather = silver_weather.dropna(subset=["weather_timestamp_utc"])


In [0]:
display(silver_weather.limit(10))

In [0]:
# ---------------------------------------------------------
# Part 2: Write and Optimize
# ---------------------------------------------------------
print("3. Writing to Silver Delta Table...")
silver_weather.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("aviation_project.silver_historical_weather")

print("   -> Optimizing table layout (Z-Ordering by airport and time)...")
# Z-Ordering by both airport and timestamp makes our future Gold layer JOIN lightning fast
spark.sql("OPTIMIZE aviation_project.silver_historical_weather ZORDER BY (airport_code, weather_timestamp_utc)")

final_count = spark.table("aviation_project.silver_historical_weather").count()
print(f"4. SUCCESS! Cleaned and saved {final_count} Silver weather records.")